# Cost per Million Tokens

Combines benchmark summaries (`results/*/summary.json`) with the pricing table
(`cost/gpu_pricing.yaml`) to compare **cost per 1M output tokens** across
providers for each experiment.

Requires analysis extras: `pip install -e ".[analysis]"`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../..")
from cost.cost_analysis import analyze, load_pricing, lookup_price

results_dir = Path("../../results")
pricing = load_pricing("../../cost/gpu_pricing.yaml")
summaries = sorted(results_dir.glob("*/summary.json"))
assert summaries, "Run benchmarks first to produce results/*/summary.json"

rows = []
for s in summaries:
    summ = json.loads(s.read_text())
    meta = summ.get("meta", {})
    gpu = meta.get("hardware", "A100-80GB")
    name = meta.get("name", s.parent.name)
    for provider in pricing.get("gpus", {}).get(gpu, {}):
        price = lookup_price(pricing, gpu, provider)
        rep = analyze(summ, price)
        rows.append({
            "experiment": name,
            "gpu": gpu,
            "provider": provider,
            "cost_per_Mtok_out": rep["cost_per_million_output_tokens"],
        })

cost_df = pd.DataFrame(rows)
pivot = cost_df.pivot_table(index="provider", columns="experiment", values="cost_per_Mtok_out")
pivot.plot(kind="bar", figsize=(10, 5))
plt.ylabel("USD / 1M output tokens")
plt.title("Cost per Million Output Tokens")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
cost_df